# Part 2 – CLAHE Image Preprocessing

**Branch:** `feature/clahe-image-preprocessing`


## 4. Preprocessing — CLAHE

### 4a. Apply CLAHE (originals are backed up to `images_original/`)

In [ ]:
def equalize_clahe(img_bgr, clip_limit=2.5, tile_grid_size=(8, 8)):
    """BGR image -> BGR image, CLAHE applied to the lightness channel only (colour preserved)."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

CLAHE_CLIP_LIMIT = 2.5
CLAHE_TILE_GRID = (8, 8)
ORIG_BACKUP_DIR = '/content/custom_data/images_original'

def preprocess_dataset_histogram_eq(image_dir, clip_limit=CLAHE_CLIP_LIMIT,
                                     tile_grid_size=CLAHE_TILE_GRID, backup=True):
    """Equalize every image in image_dir in place. Originals are copied to <image_dir>_original first; if that backup already exists the step is skipped so CLAHE is never applied twice."""
    paths = glob.glob(os.path.join(image_dir, '*.jpg')) + glob.glob(os.path.join(image_dir, '*.png'))
    if not paths:
        print(f"No images found in {image_dir} -- nothing to equalize.")
        return

    if backup:
        backup_dir = image_dir.rstrip('/') + '_original'
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir)
            for p in paths:
                shutil.copy2(p, os.path.join(backup_dir, os.path.basename(p)))
            print(f"Backed up {len(paths)} original images to {backup_dir}")
        else:
            print(f"Backup dir {backup_dir} already exists -- skipping backup "
                  f"(assuming equalization already ran once; re-running would double-apply CLAHE).")
            return

    n_ok, n_fail = 0, 0
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            n_fail += 1
            continue
        eq = equalize_clahe(img, clip_limit=clip_limit, tile_grid_size=tile_grid_size)
        cv2.imwrite(p, eq)
        n_ok += 1
    print(f"Histogram-equalized {n_ok} images in place in {image_dir} ({n_fail} unreadable/skipped).")

preprocess_dataset_histogram_eq('/content/custom_data/images')

# CLAHE-preprocessed version of the single demo pair uploaded above (in-memory + saved to disk),
# reused by every gallery/section below instead of looping over several dataset pairs.
demo_reference_bgr = cv2.imread(USER_REFERENCE_PATH)
demo_query_bgr  = cv2.imread(USER_QUERY_PATH)
demo_reference_pre = equalize_clahe(demo_reference_bgr, CLAHE_CLIP_LIMIT, CLAHE_TILE_GRID)
demo_query_pre  = equalize_clahe(demo_query_bgr,  CLAHE_CLIP_LIMIT, CLAHE_TILE_GRID)

DEMO_REFERENCE_PRE_PATH = '/content/user_reference_clahe.jpg'
DEMO_QUERY_PRE_PATH  = '/content/user_query_clahe.jpg'
cv2.imwrite(DEMO_REFERENCE_PRE_PATH, demo_reference_pre)
cv2.imwrite(DEMO_QUERY_PRE_PATH, demo_query_pre)
print('CLAHE applied to the demo pair ->', DEMO_REFERENCE_PRE_PATH, DEMO_QUERY_PRE_PATH)

### 4b. EDA Query Preprocessing

In [ ]:
bright_after, contrast_after = [], []
for p in sample_paths:
    b, c = image_brightness_contrast(p)   # sample_paths now point at the equalized files
    if b is not None:
        bright_after.append(b); contrast_after.append(c)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(bright_vals, bins=25, alpha=0.5, label='Before CLAHE', color='#4C72B0')
axes[0].hist(bright_after, bins=25, alpha=0.5, label='After CLAHE', color='#C44E52')
axes[0].set_title('Brightness: Before vs After'); axes[0].legend()

axes[1].hist(contrast_vals, bins=25, alpha=0.5, label='Before CLAHE', color='#4C72B0')
axes[1].hist(contrast_after, bins=25, alpha=0.5, label='After CLAHE', color='#C44E52')
axes[1].set_title('Contrast: Before vs After'); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Mean brightness  -> before CLAHE: {np.mean(bright_vals):.1f}, after CLAHE: {np.mean(bright_after):.1f}")
print(f"Mean contrast    -> before CLAHE: {np.mean(contrast_vals):.1f}, after CLAHE: {np.mean(contrast_after):.1f}")

In [ ]:
demo_reference_pre_rgb = cv2.cvtColor(demo_reference_pre, cv2.COLOR_BGR2RGB)
demo_query_pre_rgb  = cv2.cvtColor(demo_query_pre,  cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes[0, 0].imshow(demo_reference_raw);     axes[0, 0].set_title('Before CLAHE (demo -- Reference image)', fontsize=9); axes[0, 0].axis('off')
axes[0, 1].imshow(demo_reference_pre_rgb); axes[0, 1].set_title('After CLAHE (demo -- Reference image)', fontsize=9);  axes[0, 1].axis('off')
axes[1, 0].imshow(demo_query_raw);      axes[1, 0].set_title('Before CLAHE (demo -- Query image)', fontsize=9);  axes[1, 0].axis('off')
axes[1, 1].imshow(demo_query_pre_rgb);  axes[1, 1].set_title('After CLAHE (demo -- Query image)', fontsize=9);   axes[1, 1].axis('off')
plt.suptitle('Before vs After CLAHE -- Demo Pair')
plt.tight_layout()
plt.show()